# 3D Taylor-Green vortex DNS on Google Colab

This notebook runs the JAX pseudo-spectral solver on a Colab GPU, including a T4. The default run is deliberately short so the notebook is safe to execute interactively.

In [1]:
# Install the runtime dependencies. Colab normally already provides JAX with CUDA support.
%pip install -q matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import subprocess
from pathlib import Path

# Avoid reserving almost all T4 VRAM before the solver starts.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

workspace_dir = Path.cwd()
colab_dir = Path("/content")
if (workspace_dir / "taylor_green_jax.py").exists():
    repo_dir = workspace_dir
else:
    repo_dir = colab_dir / "jax_tg_project" if colab_dir.is_dir() and os.access(colab_dir, os.W_OK) else Path("/tmp/jax_tg_project")
    if not (repo_dir / "taylor_green_jax.py").exists():
        subprocess.run(["git", "clone", "https://github.com/mahanr/jax_tg_project.git", str(repo_dir)], check=True)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import jax
print("Solver directory:", repo_dir)
print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())
if not any(device.platform == "gpu" for device in jax.devices()):
    raise RuntimeError("No GPU detected. In Colab, select Runtime > Change runtime type > T4 GPU.")

Solver directory: /home/mahan/jax_tg_project
JAX version: 0.11.1
JAX devices: [CudaDevice(id=0)]


In [3]:
import jax.numpy as jnp
from taylor_green_jax import run_simulation

# T4-friendly parameters. Change these before running the next cell.
N = 128
dt = 0.005
reynolds = 1000
total_time = 1.0
save_every_time = 0.05

# Your requested diffusion time is: 10 * 2 * pi * 1000 ~= 62831.85.
# It requires 12,566,370 steps at dt=0.005, so it is kept as an explicit option.
requested_total_time = 10 * 2 * jnp.pi * 1000
print("Selected total_time:", total_time)
print("Requested total_time:", float(requested_total_time))

Selected total_time: 1.0
Requested total_time: 62831.853071795864


In [4]:
u_final, energies, diagnostics = run_simulation(
    N=N,
    dt=dt,
    reynolds=reynolds,
    total_time=total_time,
    save_every_time=save_every_time,
    return_diagnostics=True,
)

print("Final energy:", energies[-1])
print("Maximum divergence:", max(diagnostics["divergence_max"]))

Save time step: 10, Enstrophy: 3.743910e-01
Save time step: 20, Enstrophy: 3.739760e-01
Save time step: 30, Enstrophy: 3.737526e-01
Save time step: 40, Enstrophy: 3.737191e-01
Save time step: 50, Enstrophy: 3.738740e-01
Save time step: 60, Enstrophy: 3.742163e-01
Save time step: 70, Enstrophy: 3.747452e-01
Save time step: 80, Enstrophy: 3.754602e-01
Save time step: 90, Enstrophy: 3.763614e-01
Save time step: 100, Enstrophy: 3.774488e-01
Save time step: 110, Enstrophy: 3.787228e-01
Save time step: 120, Enstrophy: 3.801839e-01
Save time step: 130, Enstrophy: 3.818331e-01
Save time step: 140, Enstrophy: 3.836710e-01
Save time step: 150, Enstrophy: 3.856989e-01
Save time step: 160, Enstrophy: 3.879177e-01
Save time step: 170, Enstrophy: 3.903287e-01
Save time step: 180, Enstrophy: 3.929330e-01
Save time step: 190, Enstrophy: 3.957318e-01
Save time step: 200, Enstrophy: 3.987264e-01
Reynolds number: Re = 1000.0
Simulated time: 1.000 (200 steps) in 36.864 s
Energy trace: 0.125000 -> 0.120218

In [5]:
# Compact table of the saved diagnostics.
for time_value, enstrophy_value in zip(diagnostics["time"], diagnostics["enstrophy"]):
    print(f"time={time_value:10.5f}  enstrophy={enstrophy_value:.6e}")

time=   0.00000  enstrophy=3.750000e-01
time=   0.05000  enstrophy=3.743910e-01
time=   0.10000  enstrophy=3.739760e-01
time=   0.15000  enstrophy=3.737526e-01
time=   0.20000  enstrophy=3.737191e-01
time=   0.25000  enstrophy=3.738740e-01
time=   0.30000  enstrophy=3.742163e-01
time=   0.35000  enstrophy=3.747452e-01
time=   0.40000  enstrophy=3.754602e-01
time=   0.45000  enstrophy=3.763614e-01
time=   0.50000  enstrophy=3.774488e-01
time=   0.55000  enstrophy=3.787228e-01
time=   0.60000  enstrophy=3.801839e-01
time=   0.65000  enstrophy=3.818331e-01
time=   0.70000  enstrophy=3.836710e-01
time=   0.75000  enstrophy=3.856989e-01
time=   0.80000  enstrophy=3.879177e-01
time=   0.85000  enstrophy=3.903287e-01
time=   0.90000  enstrophy=3.929330e-01
time=   0.95000  enstrophy=3.957318e-01
time=   1.00000  enstrophy=3.987264e-01


The generated files are `taylor_green_slice.png` and `taylor_green_diagnostics.png` in `/content`. For a long production run, increase `total_time` only after confirming the short run is stable; the requested value is computationally very large even though it does not by itself increase VRAM usage.